In [0]:
dbutils.secrets.list('storage_key')

In [0]:
storage_account = "azurestoragepramod"
storage_key = dbutils.secrets.get(
    scope="storage_key",
    key="storage-key"
)
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)

In [0]:
dbutils.widgets.text("landing_area",'')
landing_area=dbutils.widgets.get("landing_area")

dbutils.widgets.text("source_name",'')
source_name=dbutils.widgets.get("source_name")


dbutils.widgets.text("file_name",'')
file_name=dbutils.widgets.get("file_name")

In [0]:
df=(
    spark.read.option("header","true").option("sep",",").csv(landing_area)
)

In [0]:
from pyspark.sql.types import *

if source_name == "employeer":
    expected_schema = StructType([
        StructField("employer_id", StringType(), True),
        StructField("employer_name", StringType(),False),
        StructField("payroll_provider", StringType(),False),
        StructField("status", StringType(),False)
    ])
if source_name == "benificiary":
    expected_schema = StructType([
        StructField("beneficiary_id", StringType(), True),
        StructField("participant_id", StringType(),False),
        StructField("beneficiary_name", StringType(),False),
        StructField("relationship", StringType(),False),
        StructField("allocation_pct", StringType(),False)
    ])
if source_name == "employement_history":
    expected_schema = StructType([
        StructField("history_id", StringType(), True),
        StructField("participant_id", StringType(),False),
        StructField("old_title", StringType(),False),
        StructField("new_title", StringType(),False),
        StructField("effective_date", StringType(),False)
    ])
if source_name == "participants":
    expected_schema = StructType([
        StructField("participant_id", StringType(), True),
        StructField("employee_number", StringType(),False),
        StructField("first_name", StringType(),False),
        StructField("last_name", StringType(),False),
        StructField("employer_id", StringType(),False),
        StructField("plan_id", StringType(),False),
        StructField("email", StringType(),False),
        StructField("mobile", StringType(),False),
        StructField("status", StringType(),False),
        StructField("department", StringType(),False),
        StructField("last_modified_date", StringType(),False)
    ])
if source_name == "payroll_callender":
    expected_schema = StructType([
        StructField("payroll_run_id", StringType(), True),
        StructField("period", StringType(),False),
        StructField("frequency", StringType(),False)
    ])
if source_name == "plan_enrollment":
    expected_schema = StructType([
        StructField("enrollment_id", StringType(), True),
        StructField("participant_id", StringType(),False),
        StructField("plan_id", StringType(),False),
        StructField("contribution_pct", StringType(),False)
    ])
if source_name == "plan_master":
    expected_schema = StructType([
        StructField("plan_id", StringType(), True),
        StructField("plan_name", StringType(),False),
        StructField("plan_type", StringType(),False),
        StructField("employer_id", StringType(),False),
        StructField("match_pct", StringType(),False)
    ])
if source_name == "eligibility":
    expected_schema = StructType([
        StructField("eligibility_id", StringType(), True),
        StructField("participant_id", StringType(),False),
        StructField("plan_id", StringType(),False),
        StructField("status", StringType(),False)
    ])


In [0]:
from pyspark.sql.types import *
expected = [f.name for f in expected_schema]
actual = [f.name for f in df.schema]

# print(expected)
# print(actual)

missing = set(expected) - set(actual)
# print(len(missing))
extra = set(actual) - set(expected)
# print(len(extra))

datatype_mismatch = []
print(type(len(missing)+len(extra)))
if len(missing)+len(extra) >0:
    if len(missing)>0:
        schema_validation="schema mismatch(missing column)"
        datatype_mismatch.append(list(missing))
    if len(extra)>0:
        schema_validation="schema mismatch(extra column)"
        datatype_mismatch.append(list(extra))
else:
    schema_validation="Success"

datatype_mismatch_str=",".join([item[0] for item in datatype_mismatch])

is_dup= spark.sql("""select count(*) from pramoddatabricks.control.bronze_audit
                  where file_name = '"""+file_name+"""'""").collect()[0][0]
if is_dup > 0:
    dup_validation="file is duplicate"
else:
    dup_validation="Success"





In [0]:
import json

if schema_validation == "Success" and dup_validation == "Success":
    output = {
        "result": "Success",
        "schema_validation": schema_validation,
        "dup_validation": dup_validation,
        "datatype_mismatch": datatype_mismatch_str
    }
else:
    output = {
        "result": "Failed",
        "schema_validation": schema_validation,
        "dup_validation": dup_validation,
        "datatype_mismatch": datatype_mismatch_str
    }
    
dbutils.notebook.exit(json.dumps(output))



